# MaaS Deploy & Test

Interactive notebook for deploying the Models-as-a-Service platform on an OpenShift cluster,
creating an API key, and running test inference.

**Prerequisites:**
- `oc` CLI installed and available
- `kubectl`, `kustomize` (5.7+), `jq` installed
- Access to an OpenShift cluster (logged in or credentials available)

**Sections:**
1. Configuration
2. Cluster Connection
3. Deploy MaaS Platform
4. Verify Deployment
5. Create API Key
6. List Models
7. Test Inference
8. Cleanup

## 1. Configuration

Adjust these parameters before running. Most defaults match `./scripts/deploy.sh` behavior.

In [ ]:
import os
import subprocess
import json
import time
import textwrap
from pathlib import Path

try:
    import requests
except ImportError:
    subprocess.check_call(["pip", "install", "requests"])
    import requests

# ── Deployment settings ─────────────────────────────────────
OPERATOR_TYPE   = "odh"          # "odh" or "rhoai"
DEPLOYMENT_MODE = "operator"     # "operator" or "kustomize"
ENABLE_TLS      = True
DEV_MODE        = False          # True → :latest images, False → :odh-stable

# Override images (leave empty for defaults)
MAAS_API_IMAGE        = ""       # e.g. "quay.io/opendatahub/maas-api:pr-123"
MAAS_CONTROLLER_IMAGE = ""       # e.g. "quay.io/opendatahub/maas-controller:pr-456"
OPERATOR_CATALOG      = ""       # custom catalog image for ODH/RHOAI

# Timeouts (seconds)
DEPLOY_TIMEOUT  = 900            # total deployment timeout
REQUEST_TIMEOUT = 60             # HTTP request timeout
TLS_VERIFY      = False          # set True if you trust the cluster certs

# ── Derived paths ──────────────────────────────────────────
SCRIPT_DIR   = Path(".").resolve()
PROJECT_ROOT = SCRIPT_DIR.parent if SCRIPT_DIR.name == "scripts" else SCRIPT_DIR
DEPLOY_SCRIPT = PROJECT_ROOT / "scripts" / "deploy.sh"

NAMESPACE = {
    "odh":   "opendatahub",
    "rhoai": "redhat-ods-applications",
}[OPERATOR_TYPE]

print(f"Project root:  {PROJECT_ROOT}")
print(f"Deploy script: {DEPLOY_SCRIPT}")
print(f"Operator:      {OPERATOR_TYPE}")
print(f"Namespace:     {NAMESPACE}")
print(f"TLS backend:   {ENABLE_TLS}")

### Helpers

In [ ]:
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


def run(cmd, check=True, capture=True, timeout=300, **kw):
    """Run a shell command and return CompletedProcess."""
    print(f"$ {cmd}")
    result = subprocess.run(
        cmd, shell=True, text=True, timeout=timeout,
        capture_output=capture, **kw,
    )
    if capture and result.stdout:
        print(result.stdout[:2000])
    if capture and result.stderr:
        print(result.stderr[:2000])
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed (exit {result.returncode}): {cmd}")
    return result


def oc(cmd, **kw):
    return run(f"oc {cmd}", **kw)


def kubectl(cmd, **kw):
    return run(f"kubectl {cmd}", **kw)


def kubectl_json(cmd):
    r = kubectl(f"{cmd} -o json")
    return json.loads(r.stdout)


def wait_for(description, check_fn, timeout=120, interval=10):
    """Poll check_fn() until it returns True or timeout expires."""
    print(f"Waiting for {description} (timeout {timeout}s)...")
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            if check_fn():
                print(f"  ✓ {description}")
                return True
        except Exception:
            pass
        time.sleep(interval)
    raise TimeoutError(f"Timed out waiting for: {description}")

---
## 2. Cluster Connection

Verify you are logged into the OpenShift cluster. If not, uncomment and fill in the `oc login` cell.

In [ ]:
# Uncomment and edit to log in:
# oc("login --server=https://api.my-cluster.example.com:6443 --token=sha256~...")
# oc("login -u kubeadmin -p <password> --server=https://api.my-cluster.example.com:6443")

In [ ]:
oc("whoami")
oc("cluster-info")
oc("version --short", check=False)

In [ ]:
# Verify required CLI tools
for tool in ["oc", "kubectl", "kustomize", "jq"]:
    r = run(f"which {tool}", check=False)
    status = "OK" if r.returncode == 0 else "MISSING"
    print(f"  {tool}: {status}")

---
## 3. Deploy MaaS Platform

This runs `scripts/deploy.sh` which:
1. Installs optional operators (cert-manager, LeaderWorkerSet)
2. Installs the policy engine (Kuadrant for ODH / RHCL for RHOAI)
3. Installs the primary operator (ODH / RHOAI)
4. Applies DSCInitialization and DataScienceCluster CRs
5. Deploys PostgreSQL (POC instance)
6. Configures TLS backend (Authorino ↔ MaaS API)
7. Installs maas-controller (CRDs + Deployment)
8. Waits for the Tenant reconciler to deploy maas-api

**Runtime:** 10-20 minutes depending on cluster speed.

In [ ]:
# Build the deploy command
deploy_cmd = str(DEPLOY_SCRIPT)
deploy_cmd += f" --operator-type {OPERATOR_TYPE}"

if DEPLOYMENT_MODE == "kustomize":
    deploy_cmd += " --deployment-mode kustomize"

if not ENABLE_TLS:
    deploy_cmd += " --disable-tls-backend"

if DEV_MODE:
    deploy_cmd += " --dev"

if OPERATOR_CATALOG:
    deploy_cmd += f" --operator-catalog {OPERATOR_CATALOG}"

if MAAS_API_IMAGE:
    deploy_cmd += f" --maas-api-image {MAAS_API_IMAGE}"

if MAAS_CONTROLLER_IMAGE:
    deploy_cmd += f" --maas-controller-image {MAAS_CONTROLLER_IMAGE}"

deploy_cmd += " --verbose"

print(f"Deploy command:\n  {deploy_cmd}")

In [ ]:
# Run deploy.sh — streams output in real time
env = os.environ.copy()
if MAAS_API_IMAGE:
    env["MAAS_API_IMAGE"] = MAAS_API_IMAGE
if MAAS_CONTROLLER_IMAGE:
    env["MAAS_CONTROLLER_IMAGE"] = MAAS_CONTROLLER_IMAGE
env["FORCE_OVERWRITE"] = "true"

proc = subprocess.Popen(
    deploy_cmd, shell=True, text=True, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    cwd=str(PROJECT_ROOT),
)

for line in proc.stdout:
    print(line, end="")

rc = proc.wait()
if rc != 0:
    print(f"\n⚠ deploy.sh exited with code {rc}")
else:
    print("\n✓ Deployment completed successfully")

---
## 4. Verify Deployment

Check that key components are running.

In [ ]:
print("=== MaaS Controller ===")
kubectl(f"get deployment maas-controller -n {NAMESPACE}", check=False)

print("\n=== MaaS API ===")
kubectl(f"get deployment maas-api -n {NAMESPACE}", check=False)

print("\n=== Pods ===")
kubectl(f"get pods -n {NAMESPACE} -l 'app.kubernetes.io/name in (maas-api,maas-controller)'", check=False)

print("\n=== Gateway ===")
kubectl("get gateway maas-default-gateway -n openshift-ingress", check=False)

print("\n=== AuthPolicy ===")
kubectl("get authpolicy -A", check=False)

print("\n=== Kuadrant ===")
kubectl("get kuadrant -A", check=False)

In [ ]:
# Discover the gateway URL
cluster_domain = kubectl(
    "get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}'"
).stdout.strip().strip("'")

GATEWAY_HOST = f"maas.{cluster_domain}"
GATEWAY_URL  = f"https://{GATEWAY_HOST}"
MAAS_API_URL = f"{GATEWAY_URL}/maas-api"

print(f"Gateway host:  {GATEWAY_HOST}")
print(f"Gateway URL:   {GATEWAY_URL}")
print(f"MaaS API URL:  {MAAS_API_URL}")

In [ ]:
# Wait for maas-api to become reachable through the gateway
def maas_api_reachable():
    try:
        r = requests.get(
            f"{MAAS_API_URL}/health",
            timeout=10, verify=TLS_VERIFY,
        )
        return r.status_code in (200, 401, 404)
    except Exception:
        return False

wait_for("MaaS API reachable via gateway", maas_api_reachable, timeout=300, interval=15)

---
## 5. Create API Key

The MaaS API uses API keys for inference authentication.

Flow:
1. Obtain an OpenShift identity token (`oc whoami -t`)
2. `POST /maas-api/v1/api-keys` with the OC token → returns a `sk-oai-...` API key
3. Use the API key (`Authorization: Bearer sk-oai-...`) for model catalog and inference calls

In [ ]:
# Get OC identity token
OC_TOKEN = oc("whoami -t").stdout.strip()
print(f"OC token obtained (length={len(OC_TOKEN)})")

oc_headers = {
    "Authorization": f"Bearer {OC_TOKEN}",
    "Content-Type": "application/json",
}

In [ ]:
# Create an API key
r = requests.post(
    f"{MAAS_API_URL}/v1/api-keys",
    headers=oc_headers,
    json={
        "name": f"notebook-test-{int(time.time())}",
        "expiresIn": "2h",
    },
    timeout=REQUEST_TIMEOUT,
    verify=TLS_VERIFY,
)

print(f"Status: {r.status_code}")
assert r.status_code in (200, 201), f"Failed to create API key: {r.status_code} {r.text}"

key_data = r.json()
API_KEY    = key_data["key"]
API_KEY_ID = key_data["id"]

print(f"API Key ID:     {API_KEY_ID}")
print(f"API Key prefix: {API_KEY[:20]}...")
print(f"Expires at:     {key_data.get('expiresAt', 'never')}")

api_key_headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

---
## 6. List Models

Query the model catalog through the MaaS API. Models are registered via `MaaSModelRef` CRDs.

In [ ]:
r = requests.get(
    f"{MAAS_API_URL}/v1/models",
    headers=oc_headers,
    timeout=REQUEST_TIMEOUT,
    verify=TLS_VERIFY,
)

print(f"Status: {r.status_code}")
models = r.json()
items = models.get("data") or models.get("models") or []

if items:
    print(f"\nFound {len(items)} model(s):\n")
    for m in items:
        ready = m.get("ready", "unknown")
        print(f"  • {m['id']}")
        print(f"    URL:   {m.get('url', 'N/A')}")
        print(f"    Ready: {ready}")
        print()
else:
    print("\nNo models registered yet.")
    print("Deploy a simulator model with:")
    print("  kustomize build docs/samples/models/simulator | kubectl apply --server-side=true -f -")

In [ ]:
# Also check MaaSModelRef CRDs directly
kubectl("get maasmodelrefs -A", check=False)
print()
kubectl("get maassubscriptions -A", check=False)

In [ ]:
# Select a model for inference
# Override MODEL_NAME here if you want a specific model
MODEL_NAME = ""  # e.g. "facebook/opt-125m" — leave empty to auto-select

if not MODEL_NAME and items:
    MODEL_NAME = items[0]["id"]

if MODEL_NAME and items:
    match = next((m for m in items if m["id"] == MODEL_NAME), None)
    MODEL_BASE_URL = match["url"].rstrip("/") if match and match.get("url") else None
else:
    MODEL_BASE_URL = None

print(f"Selected model: {MODEL_NAME or '(none)'}")
print(f"Model base URL: {MODEL_BASE_URL or '(not available)'}")

---
## 7. Test Inference

Send requests to the model's OpenAI-compatible endpoints through the gateway.

The request path is: `Client → Gateway → Kuadrant AuthPolicy → Backend model`

In [ ]:
if not MODEL_BASE_URL:
    print("⚠ No model available for inference. Deploy a model first.")
    print("  kustomize build docs/samples/models/simulator | kubectl apply --server-side=true -f -")
else:
    print(f"Model endpoint: {MODEL_BASE_URL}/v1/chat/completions")
    print(f"Using API key:  {API_KEY[:20]}...")

### 7a. Chat Completions

In [ ]:
if MODEL_BASE_URL:
    chat_url = f"{MODEL_BASE_URL}/v1/chat/completions"
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "user", "content": "Hello! What model are you?"}
        ],
        "max_tokens": 50,
    }

    print(f"POST {chat_url}")
    print(f"Payload: {json.dumps(payload, indent=2)}\n")

    r = requests.post(
        chat_url,
        headers=api_key_headers,
        json=payload,
        timeout=REQUEST_TIMEOUT,
        verify=TLS_VERIFY,
    )

    print(f"Status: {r.status_code}")
    if r.status_code == 200:
        data = r.json()
        print(f"Model:  {data.get('model')}")
        print(f"Tokens: {data.get('usage', {})}")
        for choice in data.get("choices", []):
            msg = choice.get("message", {})
            print(f"\nResponse:\n  {msg.get('content', '')}")
    elif r.status_code == 404:
        print("Chat completions endpoint not supported by this model.")
        print("Try the legacy completions endpoint below.")
    else:
        print(f"Response: {r.text[:500]}")

### 7b. Legacy Completions

In [ ]:
if MODEL_BASE_URL:
    completions_url = f"{MODEL_BASE_URL}/v1/completions"
    payload = {
        "model": MODEL_NAME,
        "prompt": "The capital of France is",
        "max_tokens": 30,
    }

    print(f"POST {completions_url}")
    print(f"Payload: {json.dumps(payload, indent=2)}\n")

    r = requests.post(
        completions_url,
        headers=api_key_headers,
        json=payload,
        timeout=REQUEST_TIMEOUT,
        verify=TLS_VERIFY,
    )

    print(f"Status: {r.status_code}")
    if r.status_code == 200:
        data = r.json()
        print(f"Model:  {data.get('model')}")
        print(f"Tokens: {data.get('usage', {})}")
        for choice in data.get("choices", []):
            print(f"\nResponse:\n  {choice.get('text', '')}")
    elif r.status_code == 404:
        print("Completions endpoint not supported by this model.")
    else:
        print(f"Response: {r.text[:500]}")

### 7c. Auth Verification

Verify that the gateway rejects unauthenticated and invalid requests.

In [ ]:
if MODEL_BASE_URL:
    test_url = f"{MODEL_BASE_URL}/v1/completions"
    test_payload = {"model": MODEL_NAME, "prompt": "test", "max_tokens": 1}

    # No auth header → expect 401
    r_no_auth = requests.post(
        test_url, json=test_payload,
        headers={"Content-Type": "application/json"},
        timeout=30, verify=TLS_VERIFY,
    )
    status = "✓" if r_no_auth.status_code == 401 else "✗"
    print(f"{status} No auth header  → HTTP {r_no_auth.status_code} (expected 401)")

    # Invalid API key → expect 403
    r_bad_key = requests.post(
        test_url, json=test_payload,
        headers={"Authorization": "Bearer sk-oai-INVALID", "Content-Type": "application/json"},
        timeout=30, verify=TLS_VERIFY,
    )
    status = "✓" if r_bad_key.status_code == 403 else "✗"
    print(f"{status} Invalid API key → HTTP {r_bad_key.status_code} (expected 403)")

    # Valid API key → expect 200
    r_valid = requests.post(
        test_url, json=test_payload,
        headers=api_key_headers,
        timeout=60, verify=TLS_VERIFY,
    )
    status = "✓" if r_valid.status_code == 200 else "✗"
    print(f"{status} Valid API key   → HTTP {r_valid.status_code} (expected 200)")

---
## 8. Cleanup

Revoke the API key created for testing. Run this when you're done.

In [ ]:
if API_KEY_ID:
    r = requests.delete(
        f"{MAAS_API_URL}/v1/api-keys/{API_KEY_ID}",
        headers=oc_headers,
        timeout=30,
        verify=TLS_VERIFY,
    )
    print(f"Revoke API key {API_KEY_ID}: HTTP {r.status_code}")
    if r.status_code == 200:
        print(f"  Status: {r.json().get('status')}")
else:
    print("No API key to clean up.")